In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars timm

import polars as pl

In [3]:
!nvidia-smi

Wed Apr 15 16:34:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D6:00.0 Off |                  Off |
| 30%   35C    P8             27W /  230W |     695MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
import pandas as pd
import os

BASE_PATH = 'vindr_mammogram'
meta_path = os.path.join(BASE_PATH, 'metadata.csv')

device = pd.read_csv(meta_path)
device.head()

,SOP Instance UID,Series Instance UID,SOP Instance UID.1,Patient's Age,View Position,Image Laterality,Photometric Interpretation,Rows,Columns,Imager Pixel Spacing,...,Pixel Padding Value,Pixel Padding Range Limit,Window Center,Window Width,Rescale Intercept,Rescale Slope,Rescale Type,Window Center & Width Explanation,Manufacturer,Manufacturer's Model Name
0,d8125545210c08e1b1793a5af6458ee2,b36517b9cbbcfd286a7ae04f643af97a,d8125545210c08e1b1793a5af6458ee2,053Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1662,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
1,290c658f4e75a3f83ec78a847414297c,b36517b9cbbcfd286a7ae04f643af97a,290c658f4e75a3f83ec78a847414297c,053Y,MLO,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1664,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
2,cd0fc7bc53ac632a11643ac4cc91002a,b36517b9cbbcfd286a7ae04f643af97a,cd0fc7bc53ac632a11643ac4cc91002a,053Y,CC,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1600,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
3,71638b1e853799f227492bfb08a01491,b36517b9cbbcfd286a7ae04f643af97a,71638b1e853799f227492bfb08a01491,053Y,MLO,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1654,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
4,dd9ce3288c0773e006a294188aadba8e,d931832a0815df082c085b6e09d20aac,dd9ce3288c0773e006a294188aadba8e,042Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1580,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration


In [6]:
print(device['Series Instance UID'].value_counts().value_counts())

count
4    4972
1      29
2      22
3      13
Name: count, dtype: int64


In [7]:
mask = device['Series Instance UID'].value_counts() != 4
print(device['Series Instance UID'].value_counts()[mask])

Series Instance UID
306cd45eecb913f10ce8edb80c3bc70e    3
667b7dc83aec3111ac700cf7f4812838    3
832ed2ef19c22075bffd17826981c49a    3
7b0162b7e4aa6672aa3034cc3fae5e3b    3
63fc20550753b6e8a249422bbbafacc9    3
                                   ..
dda45e7f7e76c34ac6134c05badc577b    1
269abeb40abfed98efdc989e265f5554    1
3704207e1b40779cca4d3de6c66b186b    1
d701ea42302215d0ed0473addd8c2661    1
35c90148a6a93dc6d4114937317a409b    1
Name: count, Length: 64, dtype: int64


In [8]:
mask.head()

Series Instance UID
b36517b9cbbcfd286a7ae04f643af97a    False
d931832a0815df082c085b6e09d20aac    False
a78f4822d806b4f69ba9f0e0c68778b4    False
ca7f2ada530075dd3cf15df6ee51c835    False
bc183447730d58709da1af503d7c469c    False
Name: count, dtype: bool

In [9]:
device.shape

(20000, 21)

In [10]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [11]:
density_mapping = {'A': 0, 'B': 0, 'C': 1, 'D': 1}

# density_mapping = {'A': 0, 'B': 0, 'C': 1, 'D': 1}
df['density'] = df['cc_breast_density'].str[-1].map(density_mapping)

In [12]:
import pandas as pd

device_info = device[['SOP Instance UID', "Manufacturer's Model Name"]].copy()
device_info.columns = ['image_id', 'device_model']

df = df.merge(device_info, left_on='cc_image_id', right_on='image_id', how='left')
df = df.drop('image_id', axis=1)

device_mapping = {
    'Mammomat Inspiration': 0,
    'Planmed Nuance': 1,
    'GIOTTO CLASS': 2,
    'GIOTTO IMAGE 3DL': 2
}

df['device_id'] = df['device_model'].map(device_mapping)

print(df['device_model'].value_counts())
print(df['device_id'].value_counts())
print(df.shape)

device_model
Mammomat Inspiration    7621
Planmed Nuance          1898
GIOTTO CLASS             314
GIOTTO IMAGE 3DL         166
Name: count, dtype: int64
device_id
0    7621
1    1898
2     480
Name: count, dtype: int64
(9999, 27)


In [13]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [14]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [15]:
# Count of each finding per device
cross_tab = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    margins=True, 
    margins_name='Total'
)
# Rename columns for clarity
device_names = {0: 'Mammomat', 1: 'Planmed', 2: 'GIOTTO', 'Total': 'Total'}
cross_tab.columns = [device_names.get(c, c) for c in cross_tab.columns]
print("=== Finding Distribution per Device (Counts) ===")
print(cross_tab)

# Percentage within each device (column-wise %)
cross_tab_pct = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    normalize='columns'
).round(3) * 100
cross_tab_pct.columns = [device_names.get(c, c) for c in cross_tab_pct.columns]
print("\n=== Finding Distribution per Device (% of device) ===")
print(cross_tab_pct)

# Percentage within each finding (row-wise %)
cross_tab_row_pct = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    normalize='index'
).round(3) * 100
cross_tab_row_pct.columns = [device_names.get(c, c) for c in cross_tab_row_pct.columns]
print("\n=== Device Distribution per Finding (% of finding) ===")
print(cross_tab_row_pct)

=== Finding Distribution per Device (Counts) ===
         Mammomat  Planmed  GIOTTO  Total
finding                                  
0            5199     1255     249   6703
1            1715      505     109   2329
2              89       18      20    127
3             360       69      54    483
4              35       18      32     85
5              63       14       6     83
Total        7461     1879     470   9810

=== Finding Distribution per Device (% of device) ===
         Mammomat  Planmed  GIOTTO
finding                           
0            69.7     66.8    53.0
1            23.0     26.9    23.2
2             1.2      1.0     4.3
3             4.8      3.7    11.5
4             0.5      1.0     6.8
5             0.8      0.7     1.3

=== Device Distribution per Finding (% of finding) ===
         Mammomat  Planmed  GIOTTO
finding                           
0            77.6     18.7     3.7
1            73.6     21.7     4.7
2            70.1     14.2    15.7
3      

In [16]:
# df['finding'] = df.apply(
#     lambda row: get_combined_finding(row['cc_finding_categories'], row['mlo_finding_categories']),
#     axis=1
# )
# print(df['finding'].value_counts())

In [17]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4'})
def birads_to_binary(birads):
    if birads in ['1']:
        return 0
    # if birads in ['2']:
    #     return 1
    else:
        return 1
inbreast_df['label'] = inbreast_df['birads'].apply(birads_to_binary)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,6,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [18]:
def birads_to_binary(birads):
    if birads in ['BI-RADS 1']:
        return 0
    # if birads in ['BI-RADS 2']:
    #     return 1
    return 1 
df['label'] = df['cc_breast_birads'].apply(birads_to_binary)

In [19]:
df.head()

,study_id,laterality,merged_image_path,cc_image_id,mlo_image_id,split,cc_image_path,mlo_image_path,has_cc_bbox,has_mlo_bbox,...,device_model,device_id,finding,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,label
0,0025a5dc99fd5c742026f0b2b030d3e9,L,vindr_mammogram/mammo_processed_merged1/0025a5...,451562831387e2822923204cf8f0873e,2ddfad7286c2b016931ceccd1e2c7bbc,test,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,Mammomat Inspiration,0,0,1,0,0,0,0,0,0
1,0025a5dc99fd5c742026f0b2b030d3e9,R,vindr_mammogram/mammo_processed_merged1/0025a5...,fcf12c2803ba8dc564bf1287c0c97d9a,47c8858666bcce92bcbd57974b5ce522,test,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,Mammomat Inspiration,0,0,1,0,0,0,0,0,0
2,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,L,vindr_mammogram/mammo_processed_merged1/0028fb...,3704f91985dcbc69f6ac2803523d1ecb,7fc1f1bb8bb1a7efaf7104e49c4d8b86,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,Mammomat Inspiration,0,1,0,1,0,0,0,0,1
3,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,R,vindr_mammogram/mammo_processed_merged1/0028fb...,c4ce68631bf70949570ded31a3c69e60,16e58fc1d65fa7587247e6224ee96527,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,Mammomat Inspiration,0,1,0,1,0,0,0,0,1
4,0034765af074f93ed33d5e8399355caf,L,vindr_mammogram/mammo_processed_merged1/003476...,68f09c18925a66ef2840d4a62f237b31,b664cf1e7c968896144a3a2005cd3eb4,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,Mammomat Inspiration,0,1,0,1,0,0,0,0,1


In [20]:
df.shape

(9810, 35)

In [21]:
# df = df[df['view_position'] == 'CC']
# df = df[df['laterality'] == 'R']

In [22]:

data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [23]:
print(data['study_id'].value_counts().value_counts())

count
2    3844
1     156
Name: count, dtype: int64


In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split

views_per_study    = data.groupby('study_id').size()
complete_studies   = views_per_study[views_per_study == 2].index
incomplete_studies = views_per_study[views_per_study != 2].index

study_meta = (
    data[data['study_id'].isin(complete_studies)]
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',
        'finding':          'first',
        'device_id':        'first'
    })
    .reset_index()
)

study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)           + '_' +
    study_meta['device_id'].astype(str)
)

key_counts = study_meta['stratify_key'].value_counts()
rare_keys  = key_counts[key_counts < 2].index
study_meta['stratify_key'] = study_meta['stratify_key'].apply(
    lambda k: 'rare' if k in rare_keys else k
)

train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size    = 0.1,
    stratify     = study_meta['stratify_key'],
    random_state = 423
)

train_studies = pd.concat([
    pd.Series(train_studies),
    pd.Series(incomplete_studies)
]).reset_index(drop=True)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()

# fix incomplete in train — keep as single view (don't drop)
train_views = train_df.groupby('study_id').size()
if (train_views != 2).any():
    single_view = train_views[train_views != 2].index
    train_df    = train_df[~train_df['study_id'].isin(single_view)].copy()
    single_df   = data[data['study_id'].isin(single_view)].copy()
    train_df    = pd.concat([train_df, single_df], ignore_index=True)

print(f"Train studies: {train_df['study_id'].nunique()} | Train samples: {len(train_df)}")
print(f"Val   studies: {val_df['study_id'].nunique()}   | Val   samples: {len(val_df)}")

Train studies: 3615 | Train samples: 7074
Val   studies: 385   | Val   samples: 770


In [25]:
print(train_df.groupby('study_id').size().value_counts())

2    3459
1     156
Name: count, dtype: int64


In [26]:
test_df['label'].value_counts()

label
0    1341
1     625
Name: count, dtype: int64

In [27]:
mean, std, size, bs = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 224, 8

In [28]:

TARGET_SIZE=(512,256)
  

In [29]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import pandas as pd
from PIL import Image
import os
from torchvision import transforms
import matplotlib.pyplot as plt

import random

In [30]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.6),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.03)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [31]:
import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=None,
                scale=(0.9, 1.1),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        # Gaussian noise — simulates sensor noise, keep it very mild
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [32]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "normal", 1: "abnormal"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]

    def get_pos_neg_im_paths(self, qry_idx, qry_label, qry_density, qry_device, qry_finding):
        def _paths(mask):
            return self.df[mask & (self.df.index != qry_idx)]['merged_image_path'].tolist()

        diff_dev  = self.df['device_id']           != qry_device
        same_den  = self.df['cc_breast_density']   == qry_density
        same_find = self.df['finding']             == qry_finding
        base_pos  = self.df['label']               == qry_label
        base_neg  = self.df['label']               != qry_label

        if qry_label == 0:
            pos_paths = (
                _paths(base_pos & diff_dev & same_den) or
                _paths(base_pos & diff_dev)            or
                _paths(base_pos & same_den)            or
                _paths(base_pos)
            )
        else:
            pos_paths = (
                _paths(base_pos & diff_dev & same_find) or
                _paths(base_pos & same_find)
            )

        neg_paths = (
            _paths(base_neg & diff_dev & same_den) or
            _paths(base_neg & diff_dev)            or
            _paths(base_neg & same_den)            or
            _paths(base_neg)
        )

        return random.choice(pos_paths), random.choice(neg_paths)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        qry_label   = row['label']
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
        qry_device  = row['device_id']

        qry_im = Image.open(row['merged_image_path']).convert("RGB")

        if self.transform:
            qry_im = self.transform(qry_im)

        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )

        return {
            "qry_im":      qry_im,
            "qry_gt":      torch.tensor(qry_label,   dtype=torch.long),
            "finding":     qry_finding,
            "finding_vec": finding_vec,
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long)
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)

    labels                       = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))
    print("Device distribution:\n",  train_df['device_id'].value_counts())

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, sampler=sampler,
        drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [33]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

Class counts: {np.int64(0): np.int64(4829), np.int64(1): np.int64(2245)}
Smoothed class weights: [0.811 1.189]
Device distribution:
 device_id
0    5379
1    1340
2     355
Name: count, dtype: int64


In [ ]:
import os
import gc
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# Config
# =============================================================================

FINDING_COLS = [
    "has_neg_b1",      # finding 0
    "has_neg_b2",      # finding 1
    "has_mass",        # finding 2
    "has_calc",        # finding 3
    "has_structural",  # finding 4
    "has_lymph",       # finding 5
]
NUM_FINDINGS = 6


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# =============================================================================
# Attention Pooling
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        # x: [B, C, H, W]
        w = F.softmax(self.attn(x).flatten(2), dim=-1)   # [B,1,HW]
        return (x.flatten(2) * w).sum(-1)                # [B,C]


# =============================================================================
# Query Encoder / Backbone Model
#   - No finding head
#   - No finding-conditioned classifier
#   - Standard classifier + projection head
# =============================================================================

class FindingAwareMoCoModel(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
        num_classes=2,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn = True

        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn = True

        elif "resnet" in self.backbone_name:
            self.num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.is_cnn = True

        elif "densenet" in self.backbone_name:
            self.num_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True

        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head = nn.Identity()
            self.is_cnn = False

        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.cls_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

        self.proj_head = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.BatchNorm1d(self.num_features),
            nn.GELU(),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for m in list(self.cls_head.modules()) + list(self.proj_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                if hasattr(m, "weight") and m.weight is not None:
                    nn.init.ones_(m.weight)
                if hasattr(m, "bias") and m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]

        if self.is_cnn:
            if f.ndim == 4:
                f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)

        return f

    def forward(self, x, return_embedding=False):
        feat = self._extract(x)               # [B, D]
        logits = self.cls_head(feat)          # [B, 2]

        if return_embedding:
            emb = self.proj_head(feat)
            emb = F.normalize(emb, dim=1)
            return logits, emb

        return logits


# =============================================================================
# MoCo Wrapper
#   - query encoder trained by gradients
#   - key encoder updated by EMA
#   - finding-aware queue: [num_findings, queue_size, emb_dim]
# =============================================================================

class FindingAwareMoCo(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.4,
        num_classes=2,
        num_findings=NUM_FINDINGS,
        queue_size=32,
        m=0.999,
        temperature=0.07,
    ):
        super().__init__()

        self.num_findings = num_findings
        self.queue_size = queue_size
        self.m = m
        self.temperature = temperature
        self.emb_dim = emb_dim

        # query encoder
        q_backbone = backbone_fn(weights=backbone_weights)
        self.query_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=q_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        # key encoder
        k_backbone = backbone_fn(weights=backbone_weights)
        self.key_encoder = FindingAwareMoCoModel(
            backbone_name=backbone_name,
            backbone=k_backbone,
            emb_dim=emb_dim,
            dropout=dropout,
            num_classes=num_classes,
        )

        self.key_encoder.load_state_dict(self.query_encoder.state_dict())

        for p in self.key_encoder.parameters():
            p.requires_grad = False

        # queues: one queue per finding
        self.register_buffer(
            "queues",
            F.normalize(torch.randn(num_findings, queue_size, emb_dim), dim=-1)
        )
        self.register_buffer(
            "queue_ptr",
            torch.zeros(num_findings, dtype=torch.long)
        )

    @torch.no_grad()
    def momentum_update_key_encoder(self):
        for param_q, param_k in zip(
            self.query_encoder.parameters(),
            self.key_encoder.parameters()
        ):
            param_k.data = param_k.data * self.m + param_q.data * (1.0 - self.m)

    @torch.no_grad()
    def dequeue_and_enqueue(self, keys, finding_vec):
        """
        keys: [B, D]
        finding_vec: [B, F] multi-hot
        each sample can be inserted into multiple queues
        """
        B = keys.size(0)

        for i in range(B):
            active = torch.where(finding_vec[i] > 0.5)[0]
            if len(active) == 0:
                continue

            k = keys[i]

            for f in active:
                f = int(f.item())
                ptr = int(self.queue_ptr[f].item())
                self.queues[f, ptr] = k
                self.queue_ptr[f] = (ptr + 1) % self.queue_size

    def forward_query(self, x, return_embedding=False):
        return self.query_encoder(x, return_embedding=return_embedding)

    @torch.no_grad()
    def forward_key(self, x):
        _, k = self.key_encoder(x, return_embedding=True)
        return k


# =============================================================================
# Finding-aware MoCo Loss
#
# For each anchor:
#   positives = queues of its active findings
#   negatives = all other finding queues
#
# If sample belongs to multiple findings:
#   positive pool = union of those finding queues
# =============================================================================

class FindingAwareMoCoLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, q, queues, finding_vec):
        """
        q: [B, D] normalized
        queues: [F, K, D] normalized
        finding_vec: [B, F] multi-hot
        """
        B, D = q.shape
        F_, K, D_ = queues.shape
        assert D == D_

        losses = []

        for i in range(B):
            qi = q[i]  # [D]
            active_findings = torch.where(finding_vec[i] > 0.5)[0]

            if len(active_findings) == 0:
                continue

            pos_bank = queues[active_findings]  # [n_active, K, D]
            pos_bank = pos_bank.reshape(-1, D)  # [P, D]

            inactive_findings = torch.tensor(
                [f for f in range(F_) if f not in active_findings.tolist()],
                device=q.device,
                dtype=torch.long,
            )

            if len(inactive_findings) > 0:
                neg_bank = queues[inactive_findings].reshape(-1, D)  # [N, D]
            else:
                neg_bank = None

            pos_logits = torch.matmul(pos_bank, qi) / self.tau  # [P]

            if neg_bank is not None and neg_bank.numel() > 0:
                neg_logits = torch.matmul(neg_bank, qi) / self.tau  # [N]
                logits = torch.cat([pos_logits, neg_logits], dim=0)  # [P+N]
            else:
                logits = pos_logits

            log_denom = torch.logsumexp(logits, dim=0)
            loss_i = -(pos_logits - log_denom).mean()
            losses.append(loss_i)

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device)

        return torch.stack(losses).mean()


# =============================================================================
# Classification Loss
# =============================================================================

class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=1.0, gamma_neg=2.0, alpha=0.75):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
        self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.long()
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)

        gamma = torch.where(
            targets == 1,
            torch.full_like(ce, self.gp),
            torch.full_like(ce, self.gn),
        )
        alpha_t = torch.where(
            targets == 1,
            torch.full_like(ce, self.alpha),
            torch.full_like(ce, 1.0 - self.alpha),
        )
        return (alpha_t * ((1.0 - pt) ** gamma) * ce).mean()


# =============================================================================
# Validation / Test
# =============================================================================

@torch.no_grad()
def validate(model, loader, device, cls_loss_fn, class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()

    total_loss = 0.0
    preds, targets = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        total_loss += cls_loss_fn(logits, labels).item()

        pred = logits.argmax(1)
        preds.extend(pred.cpu().numpy())
        targets.extend(labels.cpu().numpy())

    val_f1 = f1_score(targets, preds, average="binary", pos_label=1, zero_division=0)

    print("\n--- Validation ---")
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))

    return total_loss / max(len(loader), 1), val_f1


@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()

    preds, labels = [], []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        gt = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_query(imgs, return_embedding=False)
        pred = logits.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(gt.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)
    cm = confusion_matrix(labels, preds)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{'='*70}")
    print(f"{name} | Acc={acc:.4f}  F1={f1:.4f}  Sens={sens:.4f}  Spec={spec:.4f}")
    print(cm)
    print(classification_report(labels, preds, target_names=class_names, zero_division=0))
    print('=' * 70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f}  F1={f1:.4f}  Sens={sens:.4f}  Spec={spec:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(labels, preds, target_names=class_names, zero_division=0))

    return acc, f1


# =============================================================================
# Training Loop
# =============================================================================

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=1e-4,
    lr_heads=1e-4,
    epochs=60,
    patience=15,
    save_path="best_model.pth",
    con_weight=0.3,
    warmup_epochs=3,
    ramp_epochs=3,
    class_names=None,
):
    class_names = class_names or ["Benign", "Malignant"]

    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    cls_loss_fn = AsymmetricFocalLoss(
        gamma_pos=1.0,
        gamma_neg=2.0,
        alpha=0.75,
    ).to(device)

    con_loss_fn = FindingAwareMoCoLoss(
        temperature=model.temperature
    ).to(device)

    optimizer = AdamW([
        {
            "params": model.query_encoder.backbone.parameters(),
            "lr": lr_backbone,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.cls_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
        {
            "params": model.query_encoder.proj_head.parameters(),
            "lr": lr_heads,
            "weight_decay": 0.05,
        },
    ])

    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_f1 = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            cw = 0.0
        else:
            cw = con_weight * min(
                1.0,
                (epoch - warmup_epochs + 1) / max(ramp_epochs, 1)
            )

        model.train()
        e_cls = 0.0
        e_con = 0.0

        all_preds = []
        all_labels = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            imgs = batch["qry_im"].to(device, non_blocking=True)
            binary_labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                logits, q = model.forward_query(imgs, return_embedding=True)

                cls_loss = cls_loss_fn(logits, binary_labels)

                if cw > 0:
                    con_loss = con_loss_fn(q, model.queues, finding_vec)
                else:
                    con_loss = torch.tensor(0.0, device=device)

                total_loss = cls_loss + cw * con_loss

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.query_encoder.parameters(), 1.0)
                optimizer.step()

            # update momentum encoder + enqueue current batch keys
            with torch.no_grad():
                model.momentum_update_key_encoder()
                k = model.forward_key(imgs)
                model.dequeue_and_enqueue(k, finding_vec)

            pred = logits.argmax(1)
            all_preds.extend(pred.detach().cpu().numpy())
            all_labels.extend(binary_labels.detach().cpu().numpy())

            e_cls += cls_loss.item()
            e_con += con_loss.item()

            pbar.set_postfix(
                cls=f"{cls_loss.item():.3f}",
                con=f"{con_loss.item():.3f}",
                cw=f"{cw:.2f}",
            )

        n = max(len(train_loader), 1)

        print(f"\n{'='*70}")
        print(f"Epoch {epoch+1}/{epochs}  cls={e_cls/n:.4f}  con={e_con/n:.4f}  cw={cw:.4f}")
        print("\n--- Train ---")
        print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))

        val_loss, val_f1 = validate(
            model=model,
            loader=val_loader,
            device=device,
            cls_loss_fn=cls_loss_fn,
            class_names=class_names,
        )

        print(f"Val loss={val_loss:.4f}  Val F1={val_f1:.4f}")
        print('=' * 70)

        scheduler.step(epoch + 1)

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch+1}  "
                f"cls={e_cls/n:.4f}  con={e_con/n:.4f}  "
                f"cw={cw:.4f}  val_f1={val_f1:.4f}\n"
            )

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_f1": best_f1,
            }, save_path)
            print(f"Saved best model with F1={best_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_f1


# =============================================================================
# Runner
# =============================================================================

def run_experiments(train_loader, val_loader, test_loader, inbreast_loader):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    config = dict(
        lr_backbone=1e-4,
        lr_heads=1e-4,
        epochs=60,
        patience=15,
        con_weight=0.30,
        warmup_epochs=3,
        ramp_epochs=3,
        class_names=["Benign", "Malignant"],
    )

    backbones = [
        {
            "name": "efficientnet_b3",
            "fn": models.efficientnet_b3,
            "w": models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn": models.convnext_base,
            "w": models.ConvNeXt_Base_Weights.DEFAULT,
        },
    ]

    all_results = {}

    for cfg in backbones:
        out_dir = f"/workspace/Thesis_updated_results/binary_512_finding_moco/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(cfg["name"])
        print(f"{'#'*70}")

        model = FindingAwareMoCo(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.4,
            num_classes=2,
            num_findings=NUM_FINDINGS,
            queue_size=32,
            m=0.999,
            temperature=0.07,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable Params: {num_params:,}")

        best_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1} | best F1={ckpt['best_f1']:.4f}")

        acc_v, f1_v = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=config["class_names"],
        )

        acc_i, f1_i = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=config["class_names"],
        )

        all_results[cfg["name"]] = dict(
            best_val_f1=best_f1,
            vindr_acc=acc_v,
            vindr_f1=f1_v,
            inbreast_acc=acc_i,
            inbreast_f1=f1_i,
        )

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print(f"{'='*70}")
    print(f"{'Model':<22} {'ValF1':>8} {'VinDr-F1':>10} {'INbreast-F1':>13}")
    print("-" * 60)

    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_f1']:>8.4f} "
            f"{r['vindr_f1']:>10.4f} "
            f"{r['inbreast_f1']:>13.4f}"
        )

    return all_results


results = run_experiments(tr_dl, val_dl, ts_dl, inbreast_dl)

Device: cuda

######################################################################
efficientnet_b3
######################################################################
Trainable Params: 14,636,843


Epoch 1/60: 100%|██████████| 442/442 [06:16<00:00,  1.17it/s, cls=0.120, con=0.000, cw=0.00]


Epoch 1/60  cls=0.1069  con=0.0000  cw=0.0000

--- Train ---
              precision    recall  f1-score   support

      Benign       0.70      0.14      0.23      4214
   Malignant       0.42      0.91      0.57      2858

    accuracy                           0.45      7072
   macro avg       0.56      0.52      0.40      7072
weighted avg       0.58      0.45      0.37      7072




--- Validation ---
              precision    recall  f1-score   support

      Benign       0.89      0.27      0.41       533
   Malignant       0.36      0.93      0.52       237

    accuracy                           0.47       770
   macro avg       0.63      0.60      0.47       770
weighted avg       0.73      0.47      0.45       770

Val loss=0.0763  Val F1=0.5195
Saved best model with F1=0.5195


Epoch 2/60: 100%|██████████| 442/442 [05:19<00:00,  1.38it/s, cls=0.054, con=0.000, cw=0.00]


...


Epoch 23/60  cls=0.0259  con=4.1915  cw=0.3000

--- Train ---
              precision    recall  f1-score   support

      Benign       0.97      0.92      0.94      4158
   Malignant       0.89      0.96      0.92      2914

    accuracy                           0.94      7072
   macro avg       0.93      0.94      0.93      7072
weighted avg       0.94      0.94      0.94      7072


--- Validation ---
              precision    recall  f1-score   support

      Benign       0.86      0.88      0.87       533
   Malignant       0.72      0.69      0.70       237

    accuracy                           0.82       770
   macro avg       0.79      0.78      0.79       770
weighted avg       0.82      0.82      0.82       770

Val loss=0.1757  Val F1=0.7041


Epoch 24/60: 100%|██████████| 442/442 [05:41<00:00,  1.30it/s, cls=0.019, con=4.129, cw=0.30]



Epoch 24/60  cls=0.0234  con=4.1727  cw=0.3000

--- Train ---
              precision    recall  f1-score   support

      Benign       0.97      0.92      0.95      4196
   Malignant       0.90      0.96      0.93      2876

    accuracy                           0.94      7072
   macro avg       0.94      0.94      0.94      7072
weighted avg       0.94      0.94      0.94      7072


--- Validation ---
              precision    recall  f1-score   support

      Benign       0.86      0.89      0.87       533
   Malignant       0.72      0.67      0.70       237

    accuracy                           0.82       770
   macro avg       0.79      0.78      0.78       770
weighted avg       0.82      0.82      0.82       770

Val loss=0.1954  Val F1=0.6958


Epoch 25/60: 100%|██████████| 442/442 [05:32<00:00,  1.33it/s, cls=0.055, con=4.199, cw=0.30]



Epoch 25/60  cls=0.0218  con=4.1648  cw=0.3000

--- Train ---
              precision    recall  f1-score   support

      Benign       0.98      0.93      0.95      4233
   Malignant       0.90      0.97      0.93      2839

    accuracy                           0.95      7072
   macro avg       0.94      0.95      0.94      7072
weighted avg       0.95      0.95      0.95      7072



In [ ]:
dddddddddddddd

In [ ]:
import os
import gc
import math
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

FINDING_COLS      = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc",
                     "has_structural", "has_lymph"]
NUM_FINDINGS      = 6
FINDING_TO_BINARY = [0, 1, 1, 1, 1, 1]


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# ─────────────────────────────────────────────────────────────────────────────
# Backbone
# ─────────────────────────────────────────────────────────────────────────────

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, 1, bias=False),
            nn.BatchNorm2d(h), nn.GELU(),
            nn.Conv2d(h, 1, 1),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


class FindingAwareModel(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128,
                 dropout=0.4, num_classes=2):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone      = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features   = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn         = True
        elif "convnext" in self.backbone_name:
            self.num_features   = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn         = True
        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head     = nn.Identity()
            self.is_cnn       = False
        else:
            raise ValueError(backbone_name)

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        # Binary classification head
        self.classifier_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

        # Projection head for contrastive space (2-layer MLP as per SupCon paper)
        # Smaller emb_dim=128 is standard for SupCon — enough for contrastive geometry
        self.proj_head = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.BatchNorm1d(self.num_features), nn.GELU(),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for m in list(self.classifier_head.modules()) + list(self.proj_head.modules()):
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]
        if self.is_cnn:
            if f.ndim == 4:
                f = self.pool(f) if self.pool else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)
        return f

    def forward(self, x, return_embedding=False):
        f      = self._extract(x)
        logits = self.classifier_head(f)
        if return_embedding:
            emb = F.normalize(self.proj_head(f), dim=1)
            return logits, emb
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# Memory Bank  (FIFO queue — no momentum encoder needed for supervised setting)
#
# Stores past embeddings + their labels.
# Each batch: compute contrastive loss of current samples vs full queue.
# Queue updated after loss backward — embeddings from current batch enqueued.
# Queue is detached from gradient graph — gradients only flow through
# current batch embeddings.
#
# Queue size Q=4096: at batch_size=16, stores ~256 past batches.
# Even has_lymph (rare) will have 100+ queue entries after a few epochs.
# ─────────────────────────────────────────────────────────────────────────────

class MemoryBank(nn.Module):
    def __init__(self, emb_dim=128, queue_size=4096,
                 num_findings=NUM_FINDINGS, device=None):
        super().__init__()
        self.D = emb_dim
        self.Q = queue_size
        self.F = num_findings
        _d = device or torch.device("cpu")

        # Queue buffers — all initialised as zeros (unfilled)
        self.register_buffer("emb_queue",
            torch.zeros(queue_size, emb_dim, device=_d))
        self.register_buffer("binary_queue",
            torch.full((queue_size,), -1, dtype=torch.long, device=_d))
        self.register_buffer("finding_queue",
            torch.zeros(queue_size, num_findings, device=_d))
        self.register_buffer("ptr",
            torch.zeros(1, dtype=torch.long, device=_d))
        self.register_buffer("filled",
            torch.zeros(1, dtype=torch.long, device=_d))

    @torch.no_grad()
    def enqueue(self, emb: torch.Tensor,
                binary_labels: torch.Tensor,
                finding_vec: torch.Tensor):
        """FIFO enqueue. Overwrites oldest entries."""
        B = emb.shape[0]
        ptr = int(self.ptr)

        # Wrap-around FIFO
        if ptr + B <= self.Q:
            self.emb_queue[ptr:ptr+B]     = emb.detach()
            self.binary_queue[ptr:ptr+B]  = binary_labels.detach()
            self.finding_queue[ptr:ptr+B] = finding_vec.detach()
        else:
            # Split across wrap point
            tail = self.Q - ptr
            self.emb_queue[ptr:]    = emb[:tail].detach()
            self.binary_queue[ptr:] = binary_labels[:tail].detach()
            self.finding_queue[ptr:]= finding_vec[:tail].detach()
            head = B - tail
            self.emb_queue[:head]    = emb[tail:].detach()
            self.binary_queue[:head] = binary_labels[tail:].detach()
            self.finding_queue[:head]= finding_vec[tail:].detach()

        self.ptr[0]    = (ptr + B) % self.Q
        self.filled[0] = min(self.filled[0] + B, self.Q)

    def get_filled(self):
        """Return only the filled portion of the queue."""
        n = int(self.filled)
        return (self.emb_queue[:n],
                self.binary_queue[:n],
                self.finding_queue[:n])

    def stats(self):
        _, bl, fv = self.get_filled()
        n = len(bl)
        if n == 0:
            return "Queue empty"
        s = [f"total={n}"]
        s += [f"bin0={int((bl==0).sum())} bin1={int((bl==1).sum())}"]
        for f in range(self.F):
            s.append(f"f{f}={int(fv[:,f].sum())}")
        return "  ".join(s)


# ─────────────────────────────────────────────────────────────────────────────
# Hierarchical SupCon Loss with Memory Bank
#
# For each anchor z_i in the current batch:
#
# Finding-level SupCon:
#   Positives P_f(i) = queue samples sharing ANY active finding with anchor i
#   Negatives = queue samples with DIFFERENT binary class from anchor i
#   → Pulls same-finding embeddings together, pushes different-class apart
#
# Binary-level SupCon:
#   Positives P_b(i) = queue samples with same binary class as anchor i
#   Negatives = queue samples with different binary class
#   → Pulls all malignant together (cohesion), pushes benign/malignant apart
#   Weight λ < 1 so finding-level is the primary signal
#
# Combined: finding structure is maintained (mass≠calc) while binary
# separability is enforced (malignant group stays away from benign).
# ─────────────────────────────────────────────────────────────────────────────

class HierarchicalSupConLoss(nn.Module):
    def __init__(self, temperature=0.07, binary_weight=0.3,
                 finding_to_binary=None):
        super().__init__()
        self.tau          = temperature
        self.binary_weight = binary_weight
        f2b = finding_to_binary or FINDING_TO_BINARY
        self.register_buffer("f2b", torch.tensor(f2b, dtype=torch.long))

    def _supcon(self, anchors, queue_emb, pos_mask, neg_mask):
        """
        Core SupCon computation.
        anchors  : [B, D]
        queue_emb: [Q, D]
        pos_mask : [B, Q]  bool — which queue entries are positives
        neg_mask : [B, Q]  bool — which queue entries are negatives
                           (denominator = pos + neg)
        Returns scalar loss. Skips anchors with zero positives.
        """
        # Similarity of each anchor to each queue entry
        sim = anchors @ queue_emb.t() / self.tau       # [B, Q]

        # Denominator: sum over positives + negatives
        denom_mask = pos_mask | neg_mask               # [B, Q]
        # Subtract max for numerical stability
        sim_max    = sim.detach().max(dim=1, keepdim=True).values
        exp_sim    = torch.exp(sim - sim_max)

        log_denom  = torch.log(
            (exp_sim * denom_mask.float()).sum(dim=1).clamp(min=1e-8)
        ).unsqueeze(1)                                 # [B, 1]

        # Numerator: sum over positives
        n_pos = pos_mask.float().sum(dim=1)            # [B]
        has_pos = n_pos > 0

        if not has_pos.any():
            return torch.tensor(0.0, device=anchors.device)

        log_prob = sim - sim_max - log_denom           # [B, Q]
        loss_per = -(pos_mask.float() * log_prob).sum(1) \
                   / n_pos.clamp(min=1)                # [B]

        return loss_per[has_pos].mean()

    def forward(self, emb, binary_labels, finding_vec, memory_bank):
        """
        emb          : [B, D]  L2-normalised, current batch
        binary_labels: [B]     long {0,1}
        finding_vec  : [B, F]  multi-hot float
        memory_bank  : MemoryBank
        """
        device = emb.device
        q_emb, q_bin, q_fv = memory_bank.get_filled()

        if len(q_emb) == 0:
            zero = torch.tensor(0.0, device=device)
            return zero, {"finding_con": 0.0, "binary_con": 0.0}

        q_emb = q_emb.detach()        # [Q, D]  no grad through queue
        B     = emb.shape[0]
        Q     = q_emb.shape[0]

        # ── Finding-level masks ───────────────────────────────────────────────
        # Positive: queue sample shares at least one active finding with anchor
        # finding_vec[i]: [F],  q_fv[j]: [F]
        # overlap[i,j] = (finding_vec[i] * q_fv[j]).sum() > 0
        overlap = (finding_vec.unsqueeze(1) *
                   q_fv.unsqueeze(0)).sum(dim=-1)      # [B, Q]
        f_pos_mask = overlap > 0.5                     # [B, Q] bool

        # Negative: different binary class
        bin_expanded = binary_labels.unsqueeze(1)      # [B, 1]
        q_bin_exp    = q_bin.unsqueeze(0)              # [1, Q]
        neg_mask     = bin_expanded != q_bin_exp       # [B, Q] bool

        loss_finding = self._supcon(emb, q_emb, f_pos_mask, neg_mask)

        # ── Binary-level masks ────────────────────────────────────────────────
        # Positive: same binary class
        b_pos_mask = bin_expanded == q_bin_exp         # [B, Q] bool
        # Negative: different binary class (same neg_mask)

        loss_binary = self._supcon(emb, q_emb, b_pos_mask, neg_mask)

        total = loss_finding + self.binary_weight * loss_binary

        stats = {
            "finding_con": float(loss_finding.detach()),
            "binary_con":  float(loss_binary.detach()),
        }
        return total, stats


# ─────────────────────────────────────────────────────────────────────────────
# Classification Loss
# ─────────────────────────────────────────────────────────────────────────────

class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=1.0, gamma_neg=2.0, alpha=0.75):
        super().__init__()
        self.gp = gamma_pos; self.gn = gamma_neg; self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.long()
        ce      = F.cross_entropy(logits, targets, reduction="none")
        pt      = torch.exp(-ce)
        gamma   = torch.where(targets == 1, torch.full_like(ce, self.gp),
                              torch.full_like(ce, self.gn))
        alpha_t = torch.where(targets == 1, torch.full_like(ce, self.alpha),
                              torch.full_like(ce, 1 - self.alpha))
        return (alpha_t * (1 - pt) ** gamma * ce).mean()


# ─────────────────────────────────────────────────────────────────────────────
# Validate / Test
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def validate(model, loader, device, cls_loss_fn, class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()
    preds, targets, total_loss = [], [], 0.0
    for batch in loader:
        imgs   = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True)
        logits = model(imgs)
        total_loss += cls_loss_fn(logits, labels).item()
        preds.extend(logits.argmax(1).cpu().numpy())
        targets.extend(labels.cpu().numpy())
    val_f1 = f1_score(targets, preds, average="binary", pos_label=1, zero_division=0)
    print("\n--- Validation ---")
    print(classification_report(targets, preds, target_names=class_names, zero_division=0))
    return total_loss / max(len(loader), 1), val_f1, targets, preds


@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or ["Benign", "Malignant"]
    model.eval()
    preds, labels = [], []
    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        gt   = batch["qry_gt"].to(device, non_blocking=True)
        preds.extend(model(imgs).argmax(1).cpu().numpy())
        labels.extend(gt.cpu().numpy())

    acc            = accuracy_score(labels, preds)
    f1             = f1_score(labels, preds, average="binary", pos_label=1, zero_division=0)
    cm             = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    sens           = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec           = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{'='*70}\n{name} | Acc={acc:.4f}  F1={f1:.4f}  "
          f"Sens={sens:.4f}  Spec={spec:.4f}")
    print(cm)
    print(classification_report(labels, preds,
                                target_names=class_names, zero_division=0))
    print('='*70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir,
              f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\nAcc={acc:.4f}  F1={f1:.4f}  "
                 f"Sens={sens:.4f}  Spec={spec:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(labels, preds,
                 target_names=class_names, zero_division=0))
    return acc, f1


# ─────────────────────────────────────────────────────────────────────────────
# Training Loop
# ─────────────────────────────────────────────────────────────────────────────

def train_model(
    model, train_loader, val_loader, device,
    lr_backbone=1e-4, lr_heads=1e-4, epochs=60, patience=15,
    save_path="best_model.pth",
    con_weight=0.5, temperature=0.07,
    warmup_epochs=3, ramp_epochs=3,
    class_names=None, emb_dim=128,
    queue_size=4096, binary_weight=0.3,
    finding_to_binary=None,
):
    class_names       = class_names or ["Benign", "Malignant"]
    finding_to_binary = finding_to_binary or FINDING_TO_BINARY
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    cls_loss_fn = AsymmetricFocalLoss(
        gamma_pos=1.0, gamma_neg=2.0, alpha=0.75).to(device)

    memory_bank = MemoryBank(
        emb_dim=emb_dim, queue_size=queue_size,
        num_findings=NUM_FINDINGS, device=device,
    ).to(device)

    con_loss_fn = HierarchicalSupConLoss(
        temperature=temperature,
        binary_weight=binary_weight,
        finding_to_binary=finding_to_binary,
    ).to(device)

    optimizer = AdamW([
        {"params": model.backbone.parameters(),
         "lr": lr_backbone, "weight_decay": 0.05},
        {"params": model.classifier_head.parameters(),
         "lr": lr_heads, "weight_decay": 0.05},
        {"params": model.proj_head.parameters(),
         "lr": lr_heads, "weight_decay": 0.05},
    ])
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    scaler    = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_f1, not_improved = 0.0, 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            cw = 0.0
        else:
            cw = con_weight * min(
                1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))

        model.train()
        epoch_cls = epoch_con = 0.0
        all_preds, all_labels = [], []
        stats_accum = {"finding_con": 0.0, "binary_con": 0.0}

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            imgs          = batch["qry_im"].to(device, non_blocking=True)
            binary_labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec   = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type,
                                    enabled=(scaler is not None)):
                logits, emb = model(imgs, return_embedding=True)
                cls_loss    = cls_loss_fn(logits, binary_labels)

                if cw > 0:
                    con_loss, con_stats = con_loss_fn(
                        emb, binary_labels, finding_vec, memory_bank
                    )
                    total_loss = cls_loss + cw * con_loss
                else:
                    con_loss   = torch.tensor(0.0, device=device)
                    con_stats  = {k: 0.0 for k in stats_accum}
                    total_loss = cls_loss

            if scaler:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            # Enqueue AFTER backward — uses updated embeddings
            with torch.no_grad():
                _, emb_new = model(imgs, return_embedding=True)
                memory_bank.enqueue(emb_new, binary_labels, finding_vec)

            all_preds.extend(logits.argmax(1).detach().cpu().numpy())
            all_labels.extend(binary_labels.detach().cpu().numpy())
            epoch_cls   += cls_loss.item()
            epoch_con   += con_loss.item()
            for k in stats_accum:
                stats_accum[k] += con_stats.get(k, 0.0)

            pbar.set_postfix(
                cls=f"{cls_loss.item():.3f}",
                con=f"{con_loss.item():.3f}",
                cw=f"{cw:.3f}",
            )

        n = max(len(train_loader), 1)
        for k in stats_accum:
            stats_accum[k] /= n

        print(f"\n{'='*70}")
        print(f"Epoch {epoch+1}/{epochs}  cls={epoch_cls/n:.4f}  "
              f"con={epoch_con/n:.4f}  cw={cw:.4f}")
        print(f"Memory bank: {memory_bank.stats()}")
        print(f"Loss breakdown: {stats_accum}")
        print("\n--- Train ---")
        print(classification_report(all_labels, all_preds,
                                    target_names=class_names, zero_division=0))

        val_loss, val_f1, _, _ = validate(
            model, val_loader, device, cls_loss_fn, class_names)
        print(f"Val loss={val_loss:.4f}  Val F1={val_f1:.4f}")
        print('='*70)

        scheduler.step(epoch + 1)

        with open(log_path, "a") as fh:
            fh.write(f"Epoch {epoch+1}  cls={epoch_cls/n:.4f}  "
                     f"con={epoch_con/n:.4f}  cw={cw:.4f}  "
                     f"val_f1={val_f1:.4f}\n")
            fh.write(f"Bank: {memory_bank.stats()}\n")
            fh.write(f"Breakdown: {stats_accum}\n\n")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                "epoch":              epoch,
                "model_state_dict":   model.state_dict(),
                "memory_bank_state":  memory_bank.state_dict(),
                "optimizer_state":    optimizer.state_dict(),
                "best_f1":            best_f1,
            }, save_path)
            print(f"✓ Saved  best F1={best_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stop at epoch {epoch+1}")
                break

    return best_f1


# ─────────────────────────────────────────────────────────────────────────────
# Experiment Runner
# ─────────────────────────────────────────────────────────────────────────────

def run_experiments(train_loader, val_loader, test_loader, inbreast_loader):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    config = dict(
        lr_backbone=1e-4, lr_heads=1e-4, epochs=60, patience=15,
        con_weight=0.5, temperature=0.07,
        warmup_epochs=3, ramp_epochs=3,
        class_names=["Benign", "Malignant"],
        emb_dim=128,
        queue_size=4096,
        binary_weight=0.3,
        finding_to_binary=FINDING_TO_BINARY,
    )

    backbones = [
        {"name": "efficientnet_b3",
         "cls":  models.efficientnet_b3,
         "w":    models.EfficientNet_B3_Weights.DEFAULT},
        {"name": "convnext_base",
         "cls":  models.convnext_base,
         "w":    models.ConvNeXt_Base_Weights.DEFAULT},
    ]

    all_results = {}
    for cfg in backbones:
        out_dir   = f"/workspace/Thesis_updated_results/binary_512_moco/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}\n{cfg['name']}\n{'#'*70}")

        backbone = cfg["cls"](weights=cfg["w"])
        model    = FindingAwareModel(
            cfg["name"], backbone,
            emb_dim=config["emb_dim"], dropout=0.4, num_classes=2
        ).to(device)
        print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

        best_f1 = train_model(
            model=model, train_loader=train_loader, val_loader=val_loader,
            device=device, save_path=save_path, **config
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1}  F1={ckpt['best_f1']:.4f}")

        acc_v, f1_v = test_model(model, test_loader, device, out_dir,
                                 "VinDr-Test", config["class_names"])
        acc_i, f1_i = test_model(model, inbreast_loader, device, out_dir,
                                 "INbreast",   config["class_names"])

        all_results[cfg["name"]] = dict(
            best_val_f1=best_f1,
            vindr_acc=acc_v,    vindr_f1=f1_v,
            inbreast_acc=acc_i, inbreast_f1=f1_i,
        )

        del model, backbone, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*70}\nFINAL RESULTS\n{'='*70}")
    print(f"{'Model':<22} {'ValF1':>8} {'VinDr-F1':>10} {'INbreast-F1':>13}")
    print("-" * 55)
    for name, r in all_results.items():
        print(f"{name:<22} {r['best_val_f1']:>8.4f} "
              f"{r['vindr_f1']:>10.4f} {r['inbreast_f1']:>13.4f}")

    return all_results


results = run_experiments(tr_dl, val_dl, ts_dl, inbreast_dl)